In [1]:
!pip install datarec-lib

TASK 1

In [2]:
!pip install boto3

In [3]:
import boto3

s3 = boto3.client("s3")
print(s3.list_buckets())

{'ResponseMetadata': {'RequestId': '806GYDK97205RSD5', 'HostId': 'XaHt8L8MCYLxhSm3s5JlVeNq+tMlaAjUV6tnQnDdap/al0gZfnUhKFtf+90RR14G49x/KbHUdXM4egnV7E0g72nEiIZySBww', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'XaHt8L8MCYLxhSm3s5JlVeNq+tMlaAjUV6tnQnDdap/al0gZfnUhKFtf+90RR14G49x/KbHUdXM4egnV7E0g72nEiIZySBww', 'x-amz-request-id': '806GYDK97205RSD5', 'date': 'Mon, 04 May 2026 17:36:15 GMT', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'Buckets': [{'Name': 'acharya-cui-sokolenko-mwaa', 'CreationDate': datetime.datetime(2026, 2, 24, 19, 9, 1, tzinfo=tzlocal()), 'BucketArn': 'arn:aws:s3:::acharya-cui-sokolenko-mwaa'}, {'Name': 'acharya-de300-wi26', 'CreationDate': datetime.datetime(2026, 2, 17, 6, 21, 19, tzinfo=tzlocal()), 'BucketArn': 'arn:aws:s3:::acharya-de300-wi26'}, {'Name': 'adams-agrawal-evensen-mwaa', 'CreationDate': datetime.datetime(2025, 6, 7, 20, 58, 20, tzinfo=tzlocal()), 'BucketArn': 'arn:aws:s3:::adams-

In [4]:
import os
import requests
import boto3
from botocore.exceptions import ClientError

In [5]:
MOVIELENS_URL = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
LOCAL_DATA_DIR = "data"
LOCAL_ZIP_PATH = os.path.join(LOCAL_DATA_DIR, "ml-1m.zip")

BUCKET_NAME = "ha-irene-lab03"
S3_KEY = "homework_2/data/ml-1m.zip"

In [6]:
"Check whether a file already exists in S3"
def s3_object_exists(bucket_name, s3_key):
    s3 = boto3.client("s3")

    try:
        s3.head_object(Bucket=bucket_name, Key=s3_key)
        return True
    except ClientError as e:
        if e.response["Error"]["Code"] == "404":
            return False
        raise

In [7]:
"Download MovieLens 1M dataset locally"
def download_movielens(local_path=LOCAL_ZIP_PATH):
    os.makedirs(os.path.dirname(local_path), exist_ok=True)

    print("Downloading MovieLens 1M dataset...")
    response = requests.get(MOVIELENS_URL)
    response.raise_for_status()

    with open(local_path, "wb") as f:
        f.write(response.content)

    print(f"Saved dataset locally at: {local_path}")
    return local_path

In [8]:
"Upload local file to S3"
def upload_to_s3(local_path, bucket_name, s3_key):
    s3 = boto3.client("s3")
    s3.upload_file(local_path, bucket_name, s3_key)
    print(f"Uploaded to s3://{bucket_name}/{s3_key}")

In [9]:
"Task 1 maine pipeline"
def task1_download_and_upload_dataset():
    if s3_object_exists(BUCKET_NAME, S3_KEY):
        print(f"Dataset already exists at s3://{BUCKET_NAME}/{S3_KEY}. Skipping download.")
        return

    local_path = download_movielens()
    upload_to_s3(local_path, BUCKET_NAME, S3_KEY)

In [10]:
task1_download_and_upload_dataset()

Dataset already exists at s3://ha-irene-lab03/homework_2/data/ml-1m.zip. Skipping download.


TASK 2

In [11]:
!pip install --no-cache-dir torch --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu


In [12]:
!pip install --no-cache-dir transformers

In [13]:
import zipfile
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

In [14]:
PRE1980_MOVIES_LOCAL_PATH = "data/pre1980_movies.csv"
PRE1980_EMBEDDINGS_LOCAL_PATH = "data/pre1980_movie_embeddings.npy"

PRE1980_MOVIES_S3_KEY = "homework_2/intermediate/pre1980_movies.csv"
PRE1980_EMBEDDINGS_S3_KEY = "homework_2/intermediate/pre1980_movie_embeddings.npy"

MODEL_NAME = "distilbert-base-uncased"
DEVICE = "cpu"

In [15]:
"Check whether Task 2 intermediate outputs already exist in S3"
def s3_intermediate_results_exist():
    return (
        s3_object_exists(BUCKET_NAME, PRE1980_MOVIES_S3_KEY)
        and s3_object_exists(BUCKET_NAME, PRE1980_EMBEDDINGS_S3_KEY)
    )

In [16]:
"Extract the MovieLens 1M zip file locally"
def extract_movielens_dataset(zip_path=LOCAL_ZIP_PATH):
    extract_path = "data/ml-1m"

    if not os.path.exists(extract_path):
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall("data")
        print("Extracted MovieLens dataset.")
    else:
        print("MovieLens dataset already extracted.")

    return extract_path

In [17]:
"Load the MovieLens movies.dat file into a dataframe"
def load_movies(data_path):
    movies_path = os.path.join(data_path, "movies.dat")

    movies = pd.read_csv(
        movies_path,
        sep="::",
        engine="python",
        encoding="latin-1",
        names=["MovieID", "Title", "Genres"]
    )

    return movies

In [18]:
"Filter movies released in or before 1980"
def filter_movies_pre1980(movies):
    movies = movies.copy()

    movies["Year"] = movies["Title"].str.extract(r"\((\d{4})\)")
    movies["Year"] = movies["Year"].astype(int)

    pre1980_movies = movies[movies["Year"] <= 1980].copy()
    pre1980_movies = pre1980_movies.reset_index(drop=True)

    print(f"Number of movies released in or before 1980: {len(pre1980_movies)}")

    return pre1980_movies

In [19]:
"Create text strings for each movie to feed into BERT"
def create_movie_text(movies):
    movies = movies.copy()

    movies["Genres_Text"] = movies["Genres"].str.replace("|", " ", regex=False)

    movies["Text"] = (
        movies["Title"] + ". "
        + "Genres: " + movies["Genres_Text"] + ". "
        + "Release year: " + movies["Year"].astype(str)
    )

    return movies

In [20]:
"Load the DistilBERT tokenizer and encoder"
def load_bert_model():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
    encoder.eval()

    return tokenizer, encoder

In [21]:
"Create normalized DistilBERT embeddings for a list of text strings"
@torch.no_grad()
def bert_embed(texts, tokenizer, encoder, max_len=256, batch_size=32):
    all_embeddings = []

    # manual batching loop
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        batch = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        )

        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out = encoder(**batch)

        cls = out.last_hidden_state[:, 0]
        emb = F.normalize(cls, dim=-1)

        all_embeddings.append(emb.cpu().numpy().astype("float32"))

    return np.vstack(all_embeddings)

In [22]:
"Save filtered movie metadata and BERT embeddings locally"
def save_task2_outputs(pre1980_movies, embeddings):
    os.makedirs("data", exist_ok=True)

    pre1980_movies.to_csv(PRE1980_MOVIES_LOCAL_PATH, index=False)
    np.save(PRE1980_EMBEDDINGS_LOCAL_PATH, embeddings)

    print(f"Saved movies to {PRE1980_MOVIES_LOCAL_PATH}")
    print(f"Saved embeddings to {PRE1980_EMBEDDINGS_LOCAL_PATH}")

    return PRE1980_MOVIES_LOCAL_PATH, PRE1980_EMBEDDINGS_LOCAL_PATH

In [23]:
"Upload Task 2 intermediate outputs to S3"
def upload_task2_outputs_to_s3(movies_path, embeddings_path):
    upload_to_s3(movies_path, BUCKET_NAME, PRE1980_MOVIES_S3_KEY)
    upload_to_s3(embeddings_path, BUCKET_NAME, PRE1980_EMBEDDINGS_S3_KEY)

In [24]:
"Task 2 main pipeline"
def task2_create_pre1980_embeddings():
    if s3_intermediate_results_exist():
        print("Task 2 outputs already exist in S3. Skipping embedding creation.")
        return

    data_path = extract_movielens_dataset()
    movies = load_movies(data_path)
    pre1980_movies = filter_movies_pre1980(movies)
    pre1980_movies = create_movie_text(pre1980_movies)

    tokenizer, encoder = load_bert_model()
    embeddings = bert_embed(pre1980_movies["Text"].tolist(), tokenizer, encoder)

    movies_path, embeddings_path = save_task2_outputs(pre1980_movies, embeddings)
    upload_task2_outputs_to_s3(movies_path, embeddings_path)

    print("Task 2 completed.")

In [25]:
task2_create_pre1980_embeddings()

Task 2 outputs already exist in S3. Skipping embedding creation.


TASK 3

In [26]:
import json
from datetime import datetime

In [27]:
RATINGS_LOCAL_PATH = "data/ml-1m/ratings.dat"
USERS_LOCAL_PATH = "data/ml-1m/users.dat"

TASK3_RESULTS_LOCAL_PATH = "data/task3_recommendations_pre1980.json"
TASK3_RESULTS_S3_KEY = "homework_2/recommendations/task3_recommendations_pre1980.json"

load ratings and users:

In [28]:
def load_ratings(data_path="data/ml-1m"):
    ratings_path = os.path.join(data_path, "ratings.dat")

    ratings = pd.read_csv(
        ratings_path,
        sep="::",
        engine="python",
        names=["UserID", "MovieID", "Rating", "Timestamp"]
    )

    return ratings

In [29]:
def load_users(data_path="data/ml-1m"):
    users_path = os.path.join(data_path, "users.dat")

    users = pd.read_csv(
        users_path,
        sep="::",
        engine="python",
        names=["UserID", "Gender", "Age", "Occupation", "ZipCode"]
    )

    return users

Load task 2 intermediate results:

In [30]:
def load_task2_outputs():
    pre1980_movies = pd.read_csv(PRE1980_MOVIES_LOCAL_PATH)
    embeddings = np.load(PRE1980_EMBEDDINGS_LOCAL_PATH)

    return pre1980_movies, embeddings

choose users:

In [31]:
def choose_top_user(ratings):
    user_counts = ratings.groupby("UserID").size().reset_index(name="Num_Interactions")
    cutoff = user_counts["Num_Interactions"].quantile(0.95)

    top_users = user_counts[user_counts["Num_Interactions"] >= cutoff]
    selected_user = top_users.sample(1, random_state=42).iloc[0]

    return int(selected_user["UserID"]), int(selected_user["Num_Interactions"])

Cold-user recommendations: a user that the system has no data on

In [32]:
def recommend_for_cold_user(pre1980_movies, ratings, top_k=5):
    pre1980_ids = set(pre1980_movies["MovieID"])

    filtered_ratings = ratings[ratings["MovieID"].isin(pre1980_ids)]

    movie_stats = (
        filtered_ratings
        .groupby("MovieID")
        .agg(
            Avg_Rating=("Rating", "mean"),
            Num_Ratings=("Rating", "count")
        )
        .reset_index()
    )

    movie_stats = movie_stats[movie_stats["Num_Ratings"] >= 20]

    ranked = movie_stats.sort_values(
        by=["Avg_Rating", "Num_Ratings"],
        ascending=False
    )

    recommendations = ranked.head(top_k).merge(
        pre1980_movies,
        on="MovieID",
        how="left"
    )

    return recommendations[["MovieID", "Title", "Genres", "Avg_Rating", "Num_Ratings"]]

Top-user recommendations using embeddings

In [33]:
def recommend_for_top_user(user_id, pre1980_movies, embeddings, ratings, top_k=5):
    pre1980_ids = set(pre1980_movies["MovieID"])

    user_ratings = ratings[
        (ratings["UserID"] == user_id) &
        (ratings["MovieID"].isin(pre1980_ids))
    ]

    liked = user_ratings[user_ratings["Rating"] >= 4]

    if len(liked) == 0:
        return recommend_for_cold_user(pre1980_movies, ratings, top_k)

    movie_id_to_index = {
        movie_id: idx for idx, movie_id in enumerate(pre1980_movies["MovieID"])
    }

    liked_indices = [
        movie_id_to_index[movie_id]
        for movie_id in liked["MovieID"]
        if movie_id in movie_id_to_index
    ]

    user_profile_embedding = embeddings[liked_indices].mean(axis=0)
    user_profile_embedding = user_profile_embedding / np.linalg.norm(user_profile_embedding)

    scores = embeddings @ user_profile_embedding

    already_seen = set(user_ratings["MovieID"])

    candidate_movies = pre1980_movies.copy()
    candidate_movies["Similarity_Score"] = scores

    candidate_movies = candidate_movies[
        ~candidate_movies["MovieID"].isin(already_seen)
    ]

    recommendations = candidate_movies.sort_values(
        by="Similarity_Score",
        ascending=False
    ).head(top_k)

    return recommendations[["MovieID", "Title", "Genres", "Similarity_Score"]]

User Summaries:

In [34]:
def summarize_top_user(user_id, num_interactions, ratings, users):
    user_info = users[users["UserID"] == user_id].iloc[0]

    user_ratings = ratings[ratings["UserID"] == user_id]
    last_interaction_time = datetime.fromtimestamp(
        user_ratings["Timestamp"].max()
    ).strftime("%Y-%m-%d %H:%M:%S")

    return {
        "UserID": int(user_id),
        "Gender": user_info["Gender"],
        "Age": int(user_info["Age"]),
        "Occupation": int(user_info["Occupation"]),
        "ZipCode": str(user_info["ZipCode"]),
        "Num_Interactions": int(num_interactions),
        "Average_Rating": float(user_ratings["Rating"].mean()),
        "Last_Interaction_Time": last_interaction_time
    }

In [35]:
def summarize_cold_user():
    return {
        "UserID": None,
        "User_Type": "Cold user",
        "Last_Interaction_Time": None,
        "Summary": "No historical interaction data is available for this user."
    }

Save and upload results

In [36]:
def save_task3_results(results):
    os.makedirs("data", exist_ok=True)

    with open(TASK3_RESULTS_LOCAL_PATH, "w") as f:
        json.dump(results, f, indent=4)

    print(f"Saved Task 3 results to {TASK3_RESULTS_LOCAL_PATH}")
    return TASK3_RESULTS_LOCAL_PATH

In [37]:
def upload_task3_results_to_s3(local_path):
    upload_to_s3(local_path, BUCKET_NAME, TASK3_RESULTS_S3_KEY)

Task 3 main pipeline

In [38]:
def task3_generate_recommendations_pre1980():
    data_path = extract_movielens_dataset()

    ratings = load_ratings(data_path)
    users = load_users(data_path)
    pre1980_movies, embeddings = load_task2_outputs()

    top_user_id, num_interactions = choose_top_user(ratings)

    cold_recs = recommend_for_cold_user(pre1980_movies, ratings)
    top_user_recs = recommend_for_top_user(
        top_user_id,
        pre1980_movies,
        embeddings,
        ratings
    )

    results = [
        {
            "User_Type": "Cold user",
            "Last_Interaction_Time": None,
            "User_Summary": summarize_cold_user(),
            "Recommended_Movies": cold_recs.to_dict(orient="records")
        },
        {
            "User_Type": "Top user",
            "Last_Interaction_Time": summarize_top_user(
                top_user_id, num_interactions, ratings, users
            )["Last_Interaction_Time"],
            "User_Summary": summarize_top_user(
                top_user_id, num_interactions, ratings, users
            ),
            "Recommended_Movies": top_user_recs.to_dict(orient="records")
        }
    ]

    local_path = save_task3_results(results)
    upload_task3_results_to_s3(local_path)

    print("Task 3 completed.")

In [39]:
task3_generate_recommendations_pre1980()

MovieLens dataset already extracted.
Saved Task 3 results to data/task3_recommendations_pre1980.json
Uploaded to s3://ha-irene-lab03/homework_2/recommendations/task3_recommendations_pre1980.json
Task 3 completed.


TASK 4

In [40]:
FULL_MOVIES_LOCAL_PATH = "data/full_movies.csv"
FULL_EMBEDDINGS_LOCAL_PATH = "data/full_movie_embeddings.npy"

FULL_MOVIES_S3_KEY = "homework_2/intermediate/full_movies.csv"
FULL_EMBEDDINGS_S3_KEY = "homework_2/intermediate/full_movie_embeddings.npy"

TASK4_RESULTS_LOCAL_PATH = "data/task4_recommendations_full.json"
TASK4_RESULTS_S3_KEY = "homework_2/recommendations/task4_recommendations_full.json"

In [41]:
def full_intermediate_results_exist():
    "Check whether Task 4 full-data intermediate outputs already exist in S3"
    return (
        s3_object_exists(BUCKET_NAME, FULL_MOVIES_S3_KEY)
        and s3_object_exists(BUCKET_NAME, FULL_EMBEDDINGS_S3_KEY)
    )

In [42]:
def prepare_full_movies(movies):
    "Prepare the full movie dataset for BERT embeddings"
    movies = movies.copy()

    movies["Year"] = movies["Title"].str.extract(r"\((\d{4})\)")
    movies["Year"] = movies["Year"].astype(int)

    movies = create_movie_text(movies)
    movies = movies.reset_index(drop=True)

    print(f"Number of movies in full dataset: {len(movies)}")

    return movies

In [43]:
def save_task4_embeddings_outputs(full_movies, embeddings):
    "Save full movie metadata and BERT embeddings locally"
    os.makedirs("data", exist_ok=True)

    full_movies.to_csv(FULL_MOVIES_LOCAL_PATH, index=False)
    np.save(FULL_EMBEDDINGS_LOCAL_PATH, embeddings)

    print(f"Saved full movies to {FULL_MOVIES_LOCAL_PATH}")
    print(f"Saved full embeddings to {FULL_EMBEDDINGS_LOCAL_PATH}")

    return FULL_MOVIES_LOCAL_PATH, FULL_EMBEDDINGS_LOCAL_PATH

In [44]:
def upload_task4_embeddings_outputs_to_s3(movies_path, embeddings_path):
    "Upload Task 4 full-data intermediate outputs to S3"
    upload_to_s3(movies_path, BUCKET_NAME, FULL_MOVIES_S3_KEY)
    upload_to_s3(embeddings_path, BUCKET_NAME, FULL_EMBEDDINGS_S3_KEY)

In [45]:
def task4_create_full_embeddings():
    "Task 4 embedding pipeline using the full dataset"
    if full_intermediate_results_exist():
        print("Task 4 full-data embeddings already exist in S3. Skipping embedding creation.")
        return

    data_path = extract_movielens_dataset()
    movies = load_movies(data_path)
    full_movies = prepare_full_movies(movies)

    tokenizer, encoder = load_bert_model()
    embeddings = bert_embed(full_movies["Text"].tolist(), tokenizer, encoder)

    movies_path, embeddings_path = save_task4_embeddings_outputs(full_movies, embeddings)
    upload_task4_embeddings_outputs_to_s3(movies_path, embeddings_path)

    print("Task 4 full embedding step completed.")

for recommendations: 

In [46]:
def load_task4_outputs():
    "Load full movie metadata and embeddings"
    full_movies = pd.read_csv(FULL_MOVIES_LOCAL_PATH)
    full_embeddings = np.load(FULL_EMBEDDINGS_LOCAL_PATH)

    return full_movies, full_embeddings

In [47]:
def task4_generate_recommendations_full():
    "Task 4 recommendation pipeline using the full dataset"
    data_path = extract_movielens_dataset()

    ratings = load_ratings(data_path)
    users = load_users(data_path)
    full_movies, full_embeddings = load_task4_outputs()

    top_user_id, num_interactions = choose_top_user(ratings)

    cold_recs = recommend_for_cold_user(full_movies, ratings)
    top_user_recs = recommend_for_top_user(
        top_user_id,
        full_movies,
        full_embeddings,
        ratings
    )

    top_user_summary = summarize_top_user(top_user_id, num_interactions, ratings, users)

    results = [
        {
            "User_Type": "Cold user",
            "Last_Interaction_Time": None,
            "User_Summary": summarize_cold_user(),
            "Recommended_Movies": cold_recs.to_dict(orient="records")
        },
        {
            "User_Type": "Top user",
            "Last_Interaction_Time": top_user_summary["Last_Interaction_Time"],
            "User_Summary": top_user_summary,
            "Recommended_Movies": top_user_recs.to_dict(orient="records")
        }
    ]

    os.makedirs("data", exist_ok=True)

    with open(TASK4_RESULTS_LOCAL_PATH, "w") as f:
        json.dump(results, f, indent=4)

    print(f"Saved Task 4 recommendations to {TASK4_RESULTS_LOCAL_PATH}")

    upload_to_s3(TASK4_RESULTS_LOCAL_PATH, BUCKET_NAME, TASK4_RESULTS_S3_KEY)

    print("Task 4 full recommendation step completed.")

In [48]:
# run task 4
task4_create_full_embeddings()
task4_generate_recommendations_full()

Task 4 full-data embeddings already exist in S3. Skipping embedding creation.
MovieLens dataset already extracted.
Saved Task 4 recommendations to data/task4_recommendations_full.json
Uploaded to s3://ha-irene-lab03/homework_2/recommendations/task4_recommendations_full.json
Task 4 full recommendation step completed.


In [49]:
# double checking s3
!aws s3 ls s3://ha-irene-lab03/homework_2/intermediate/
!aws s3 ls s3://ha-irene-lab03/homework_2/recommendations/

2026-05-03 04:07:17   11928704 full_movie_embeddings.npy
2026-05-03 04:07:17     492283 full_movies.csv
2026-05-03 02:35:11    2724992 pre1980_movie_embeddings.npy
2026-05-03 02:35:11     114415 pre1980_movies.csv
2026-05-04 17:36:39       3168 task3_recommendations_pre1980.json
2026-05-04 17:36:45       3127 task4_recommendations_full.json


TASK 5

In [50]:
MY_PROFILE_LOCAL_PATH = "data/my_user_profile.json"
MY_RECOMMENDATIONS_LOCAL_PATH = "data/task5_my_recommendations.json"

MY_PROFILE_S3_KEY = "homework_2/user_profiles/my_user_profile.json"
MY_RECOMMENDATIONS_S3_KEY = "homework_2/recommendations/task5_my_recommendations.json"

In [51]:
MY_MOVIE_RATINGS = [
    {"Title": "Toy Story (1995)", "Rating": 5},
    {"Title": "Jumanji (1995)", "Rating": 3},
    {"Title": "Father of the Bride Part II (1995)", "Rating": 3},
    {"Title": "Heat (1995)", "Rating": 4},
    {"Title": "Sabrina (1995)", "Rating": 3},
    {"Title": "GoldenEye (1995)", "Rating": 4},
    {"Title": "Sense and Sensibility (1995)", "Rating": 4},
    {"Title": "Casino (1995)", "Rating": 4},
    {"Title": "Clueless (1995)", "Rating": 5},
    {"Title": "Seven (Se7en) (1995)", "Rating": 4},
]

In [52]:
"Create my user profile by matching selected movie titles to MovieLens metadata"
def create_my_user_profile(my_ratings, full_movies):
    profile_movies = []

    for item in my_ratings:
        title = item["Title"]
        rating = item["Rating"]

        match = full_movies[full_movies["Title"] == title]

        if len(match) == 0:
            print(f"Warning: movie not found: {title}")
            continue

        movie_row = match.iloc[0]

        profile_movies.append({
            "MovieID": int(movie_row["MovieID"]),
            "Title": movie_row["Title"],
            "Genres": movie_row["Genres"],
            "Year": int(movie_row["Year"]),
            "Rating": rating
        })

    profile = {
        "User_Type": "My user profile",
        "Created_At": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "Num_Rated_Movies": len(profile_movies),
        "Rated_Movies": profile_movies
    }

    return profile

In [53]:
"Recommend movies for myself using embeddings from highly rated movies"
def recommend_for_myself(my_profile, full_movies, full_embeddings, top_k=5):
    movie_id_to_index = {
        movie_id: idx for idx, movie_id in enumerate(full_movies["MovieID"])
    }

    liked_movie_ids = [
        movie["MovieID"]
        for movie in my_profile["Rated_Movies"]
        if movie["Rating"] >= 4
    ]

    liked_indices = [
        movie_id_to_index[movie_id]
        for movie_id in liked_movie_ids
        if movie_id in movie_id_to_index
    ]

    my_profile_embedding = full_embeddings[liked_indices].mean(axis=0)
    my_profile_embedding = my_profile_embedding / np.linalg.norm(my_profile_embedding)

    scores = full_embeddings @ my_profile_embedding

    already_rated = {
        movie["MovieID"]
        for movie in my_profile["Rated_Movies"]
    }

    candidate_movies = full_movies.copy()
    candidate_movies["Similarity_Score"] = scores

    candidate_movies = candidate_movies[
        ~candidate_movies["MovieID"].isin(already_rated)
    ]

    recommendations = candidate_movies.sort_values(
        by="Similarity_Score",
        ascending=False
    ).head(top_k)

    return recommendations[["MovieID", "Title", "Genres", "Similarity_Score"]]

In [54]:
"Save my user profile and recommendations locally"
def save_my_profile_and_recommendations(my_profile, recommendations):
    os.makedirs("data", exist_ok=True)

    with open(MY_PROFILE_LOCAL_PATH, "w") as f:
        json.dump(my_profile, f, indent=4)

    results = {
        "User_Type": "My user profile",
        "Last_Interaction_Time": my_profile["Created_At"],
        "User_Summary": {
            "Num_Rated_Movies": my_profile["Num_Rated_Movies"],
            "Rated_Movies": my_profile["Rated_Movies"]
        },
        "Recommended_Movies": recommendations.to_dict(orient="records")
    }

    with open(MY_RECOMMENDATIONS_LOCAL_PATH, "w") as f:
        json.dump(results, f, indent=4)

    return MY_PROFILE_LOCAL_PATH, MY_RECOMMENDATIONS_LOCAL_PATH

In [55]:
"Upload my user profile and recommendation results to S3"
def upload_task5_outputs_to_s3(profile_path, recommendations_path):
    upload_to_s3(profile_path, BUCKET_NAME, MY_PROFILE_S3_KEY)
    upload_to_s3(recommendations_path, BUCKET_NAME, MY_RECOMMENDATIONS_S3_KEY)

In [56]:
"Task 5 main pipeline"
def task5_recommend_for_myself():
    full_movies, full_embeddings = load_task4_outputs()

    my_profile = create_my_user_profile(MY_MOVIE_RATINGS, full_movies)
    recommendations = recommend_for_myself(my_profile, full_movies, full_embeddings)

    profile_path, recommendations_path = save_my_profile_and_recommendations(
        my_profile,
        recommendations
    )

    upload_task5_outputs_to_s3(profile_path, recommendations_path)

    print("Task 5 completed.")

In [57]:
task5_recommend_for_myself()

Uploaded to s3://ha-irene-lab03/homework_2/user_profiles/my_user_profile.json
Uploaded to s3://ha-irene-lab03/homework_2/recommendations/task5_my_recommendations.json
Task 5 completed.


In [58]:
!aws s3 ls s3://ha-irene-lab03/homework_2/user_profiles/
!aws s3 ls s3://ha-irene-lab03/homework_2/recommendations/

2026-05-04 17:36:48       1977 my_user_profile.json
2026-05-04 17:36:39       3168 task3_recommendations_pre1980.json
2026-05-04 17:36:45       3127 task4_recommendations_full.json
2026-05-04 17:36:48       3264 task5_my_recommendations.json


In [59]:
import json

with open("data/task5_my_recommendations.json") as f:
    data = json.load(f)

data

{'User_Type': 'My user profile',
 'Last_Interaction_Time': '2026-05-04 17:36:47',
 'User_Summary': {'Num_Rated_Movies': 10,
  'Rated_Movies': [{'MovieID': 1,
    'Title': 'Toy Story (1995)',
    'Genres': "Animation|Children's|Comedy",
    'Year': 1995,
    'Rating': 5},
   {'MovieID': 2,
    'Title': 'Jumanji (1995)',
    'Genres': "Adventure|Children's|Fantasy",
    'Year': 1995,
    'Rating': 3},
   {'MovieID': 5,
    'Title': 'Father of the Bride Part II (1995)',
    'Genres': 'Comedy',
    'Year': 1995,
    'Rating': 3},
   {'MovieID': 6,
    'Title': 'Heat (1995)',
    'Genres': 'Action|Crime|Thriller',
    'Year': 1995,
    'Rating': 4},
   {'MovieID': 7,
    'Title': 'Sabrina (1995)',
    'Genres': 'Comedy|Romance',
    'Year': 1995,
    'Rating': 3},
   {'MovieID': 10,
    'Title': 'GoldenEye (1995)',
    'Genres': 'Action|Adventure|Thriller',
    'Year': 1995,
    'Rating': 4},
   {'MovieID': 17,
    'Title': 'Sense and Sensibility (1995)',
    'Genres': 'Drama|Romance',
    